# 📄 Resume RAG System for Intelligent Job Matching

This project implements a **Retrieval-Augmented Generation (RAG)** system for semantic resume retrieval and job matching.

## Project Workflow

01. Resume PDFs
02. PDF Text Extraction
03. Intelligent Chunking
04. Metadata Extraction
05. Embedding Generation
06. Chroma Vector Database
07. Semantic Retrieval
08. Hybrid Search (BM25)
09. Candidate Ranking
10. JSON Output 

This project uses the following libraries:

| Library | Purpose |
|----------|----------|
| PyMuPDF | Read PDF resumes |
| sentence-transformers | Generate embeddings |
| ChromaDB | Vector database |
| LangChain | Text splitting |
| Rank-BM25 | Keyword search |
| spaCy | Metadata extraction |
| pandas | Data analysis |
| NumPy | Numerical operations |
| tqdm | Progress bars |

These libraries together implement the complete Retrieval-Augmented Generation (RAG) pipeline.

In [1]:
# Uncomment and run only once

!pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Download the spaCy English model (run once)
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -------------- ------------------------- 4.7/12.8 MB 52.3 MB/s eta 0:00:01
     --------------------------------- ----- 11.0/12.8 MB 35.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 26.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Required Libraries

In this section we import all libraries required for the project.

The imports are grouped according to their purpose:

- File Handling
- PDF Processing
- Machine Learning
- Embedding Models
- Vector Database
- Metadata Extraction
- Visualization

In [3]:
import os
import json
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# LangChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenRouter
from langchain_openrouter import ChatOpenRouter

# ChromaDB
from langchain_chroma import Chroma

# Hybrid Search
from rank_bm25 import BM25Okapi

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


# Configure the Environment

This project uses **OpenRouter** as the gateway for accessing Large Language Models (LLMs).

The API key is stored securely in a `.env` file located in the project root.

Example:

```text
OPENROUTER_API_KEY=your_openrouter_api_key
```

Using environment variables keeps sensitive credentials out of the source code and makes the project easier to deploy and share.

In [4]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("❌ OPENROUTER_API_KEY not found in .env file.")

print("✅ OpenRouter API Key loaded successfully.")

✅ OpenRouter API Key loaded successfully.


# Initialize AI Models

A Retrieval-Augmented Generation (RAG) system consists of two primary AI components:

### 1. Embedding Model

The embedding model converts textual data into dense numerical vectors that capture semantic meaning. These embeddings enable efficient similarity search within the vector database.

Embedding Model:
- **text-embedding-3-small**

### 2. Large Language Model (LLM)

The Large Language Model (LLM) is responsible for understanding natural language and generating human-readable responses.

In this project, the LLM is used to:

- Extract resume metadata
- Explain candidate-job matches
- Generate structured JSON responses
- Support future enhancements such as resume summarization

LLM:
- **OpenAI GPT-4o Mini** (accessed through OpenRouter)

Separating retrieval (Embeddings) from reasoning (LLM) is a common architecture used in modern RAG systems.

In [5]:
# ============================================================
# Initialize Embedding Model
# ============================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model initialized : sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model initialized : sentence-transformers/all-MiniLM-L6-v2


In [6]:
# ============================================================
# Initialize LLM
# ============================================================

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    max_tokens=150
)

print("✅ LLM initialized : openai/gpt-4o-mini")

✅ LLM initialized : openai/gpt-4o-mini


# Load Resume Documents

The first stage of a Retrieval-Augmented Generation (RAG) pipeline is **document ingestion**.

In this step, we load all resume PDF files from the `data/resumes/` directory using LangChain's **PyPDFLoader**.

Each PDF is converted into one or more **Document** objects.

A `Document` contains:

- **page_content** – The extracted text from each page.
- **metadata** – Information such as the source file path and page number.

These documents will later be:
- Chunked into smaller sections
- Converted into embeddings
- Stored in the vector database for semantic retrieval

In [7]:
# ============================================================
# Configure Project Directories
# ============================================================

from pathlib import Path

# Project Root
PROJECT_ROOT = Path.cwd().parent

# Data Directories
DATA_DIR = PROJECT_ROOT / "data"

RESUME_DIR = DATA_DIR / "resumes"

JOB_DESCRIPTION_DIR = DATA_DIR / "job_descriptions"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

VECTOR_STORE_DIR = PROJECT_ROOT / os.getenv(
    "VECTOR_STORE_DIR",
    "data/chroma_db"
)

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Resume Directory  : {RESUME_DIR}")
print(f"Vector Store      : {VECTOR_STORE_DIR}")

Project Root      : d:\GitHub\RAG-Based-Profile-matching
Resume Directory  : d:\GitHub\RAG-Based-Profile-matching\data\resumes
Vector Store      : d:\GitHub\RAG-Based-Profile-matching\data\chroma_db


In [8]:
# ============================================================
# Find Resume PDFs
# ============================================================

resume_files = sorted(RESUME_DIR.glob("*.pdf"))

print(f"Found {len(resume_files)} resume(s).\n")

for pdf in resume_files:
    print(pdf.name)

Found 34 resume(s).

alex_green.pdf
alice_smith.pdf
andrew_anderson.pdf
brandon_hernandez.pdf
brittany_white.pdf
chloe_harris.pdf
chloe_hill.pdf
chloe_jackson.pdf
chloe_wright.pdf
christopher_hall.pdf
daniel_clark.pdf
daniel_garcia.pdf
david_wilson.pdf
elena_lewis.pdf
emily_davis.pdf
ethan_young.pdf
hannah_scott.pdf
james_martinez.pdf
john_doe.pdf
justin_lee.pdf
kevin_allen.pdf
kevin_young.pdf
laura_hill.pdf
marcus_harris.pdf
marcus_lewis.pdf
michael_brown.pdf
priya_sharma.pdf
robert_chen.pdf
sarah_jenkins.pdf
sophia_taylor.pdf
victoria_martinez.pdf
victoria_walker.pdf
william_green.pdf
zoe_lewis.pdf


In [9]:
# ============================================================
# Load Resume Documents
# ============================================================

all_documents = []

for resume_path in tqdm(resume_files, desc="Loading Resumes"):

    loader = PyPDFLoader(str(resume_path))

    documents = loader.load()

    # Add custom metadata
    for doc in documents:
        doc.metadata["resume_name"] = resume_path.stem
        doc.metadata["resume_path"] = str(resume_path)

    all_documents.extend(documents)

print(f"\n✅ Total pages loaded: {len(all_documents)}")

Loading Resumes: 100%|██████████| 34/34 [00:01<00:00, 24.87it/s]


✅ Total pages loaded: 34


In [10]:
# Display one sample document

sample_doc = all_documents[0]

print("Metadata:")
print(sample_doc.metadata)

print("\nPreview:\n")
print(sample_doc.page_content[:1000])

Metadata:
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-08-08T01:08:36+05:30', 'author': 'Mohankumar Markuli Chandrayigowda', 'moddate': '2026-08-08T01:08:36+05:30', 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'resume_name': 'alex_green', 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf'}

Preview:

ALEX GREEN 
Email: alex.green14@example.com | Phone: +1-555-0114 
Location: Austin, TX 
 
PROFESSIONAL SUMMARY 
Experienced Backend Engineer with 9 years of hands-on industry experience building 
scalable systems and data products. 
 
TECHNICAL SKILLS 
Python, FastAPI, Django, PostgreSQL, Redis, Docker, AWS, GraphQL, Microservices, Linux, 
Unit Testing, REST APIs 
 
EDUCATION 
Bachelor of Science in Computer Science, UC San Diego (2015) 
 
PROFESSIONAL EXPERIENCE 
Senior Backend Engineer | Tech Company E | 2020 – Present 

In [30]:
for i, doc in enumerate(all_documents[:3]):
    print("=" * 80)
    print(f"Document {i+1}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:100])

Document 1
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-08-08T01:08:36+05:30', 'author': 'Mohankumar Markuli Chandrayigowda', 'moddate': '2026-08-08T01:08:36+05:30', 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'resume_name': 'alex_green', 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf'}
ALEX GREEN 
Email: alex.green14@example.com | Phone: +1-555-0114 
Location: Austin, TX 
 
PROFESSION
Document 2
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-08-08T01:10:22+05:30', 'author': 'Mohankumar Markuli Chandrayigowda', 'moddate': '2026-08-08T01:10:22+05:30', 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alice_smith.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'resume_name': 'alice_smith', 'resume_path': 'd:\\GitHub\\RAG-Based-P

# Intelligent Document Chunking

Large Language Models and embedding models have context length limitations, making it inefficient to embed an entire resume as a single document.

Instead, each resume is divided into smaller, overlapping chunks.

### Why Chunking?

- Improves semantic search accuracy.
- Preserves local context within each chunk.
- Enables retrieval of the most relevant sections instead of the entire resume.
- Reduces embedding computation for future updates.

### Chunking Strategy

The chunk size and overlap are configured through the `.env` file.

Configuration:

- **Chunk Size:** 1000 characters
- **Chunk Overlap:** 200 characters

The overlap helps preserve context between adjacent chunks, ensuring important information is not split abruptly.

In [41]:
# ============================================================
# Configure Text Splitter
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP")),
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

print("✅ Text splitter initialized.")

✅ Text splitter initialized.


In [42]:
# ============================================================
# Split Documents into Chunks
# ============================================================

chunked_documents = text_splitter.split_documents(all_documents)

print(f"Total Chunks Created: {len(chunked_documents)}")

Total Chunks Created: 47


In [43]:
# ============================================================
# Inspect a Sample Chunk
# ============================================================

sample_chunk = chunked_documents[0]

print("Metadata:")
print(sample_chunk.metadata)

print("\nChunk Length:")
print(len(sample_chunk.page_content))

print("\nChunk Preview:\n")
print(sample_chunk.page_content[:1000])

Metadata:
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-08-08T01:08:36+05:30', 'author': 'Mohankumar Markuli Chandrayigowda', 'moddate': '2026-08-08T01:08:36+05:30', 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'resume_name': 'alex_green', 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\alex_green.pdf'}

Chunk Length:
994

Chunk Preview:

ALEX GREEN 
Email: alex.green14@example.com | Phone: +1-555-0114 
Location: Austin, TX 
 
PROFESSIONAL SUMMARY 
Experienced Backend Engineer with 9 years of hands-on industry experience building 
scalable systems and data products. 
 
TECHNICAL SKILLS 
Python, FastAPI, Django, PostgreSQL, Redis, Docker, AWS, GraphQL, Microservices, Linux, 
Unit Testing, REST APIs 
 
EDUCATION 
Bachelor of Science in Computer Science, UC San Diego (2015) 
 
PROFESSIONAL EXPERIENCE 
Senior Backend Engineer | Tech Co

# Generate Embeddings and Create the Vector Database

Once the resume documents have been split into meaningful chunks, the next step is to convert each chunk into a numerical vector representation called an **embedding**.

Embeddings capture the semantic meaning of text, enabling similarity-based retrieval instead of exact keyword matching.

In this project, the embedding model converts every resume chunk into a dense vector, which is then stored in **ChromaDB**, a persistent vector database.

### Why ChromaDB?

- Persistent storage on disk
- Fast similarity search
- Metadata filtering
- Seamless integration with LangChain
- Lightweight and suitable for local RAG applications

At the end of this step, every resume chunk will be indexed and ready for semantic retrieval.

In [44]:
# ============================================================
# Create Chroma Vector Store
# ============================================================

vector_store = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_model,
    persist_directory=str(VECTOR_STORE_DIR)
)

print("✅ Vector Store Created Successfully")

✅ Vector Store Created Successfully


In [45]:
print(vector_store)

In [46]:
collection = vector_store._collection

print(f"Total Chunks Stored : {collection.count()}")

Total Chunks Stored : 47


In [70]:
TOP_K = 10

retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("✅ Retriever created successfully.")

✅ Retriever created successfully.


#  Load Job Description

The Resume RAG system supports two ways of providing a job description:

1. **Select an existing Job Description** from the `data/job_descriptions/` directory.
2. **Provide a custom Job Description** by entering the path to a text file.

This flexibility allows recruiters to evaluate resumes against predefined roles or new hiring requirements without modifying the code.

In [49]:
# ============================================================
# Available Job Descriptions
# ============================================================

jd_files = sorted(JOB_DESCRIPTION_DIR.glob("*.txt"))

print("Available Job Descriptions:\n")

for index, jd in enumerate(jd_files, start=1):
    print(f"{index}. {jd.stem}")

Available Job Descriptions:

1. jd_ai_product_manager
2. jd_devops_cloud_architect
3. jd_frontend_react_engineer
4. jd_fullstack_python_dev
5. jd_senior_data_engineer
6. jd_senior_ml_engineer


In [57]:
# ============================================================
# Select Job Description
# ============================================================

print("\nChoose an option:")

print("1. Select an existing Job Description")

print("2. Use your own Job Description (.txt file)")

choice = input("\nEnter your choice (1 or 2): ").strip()


Choose an option:
1. Select an existing Job Description
2. Use your own Job Description (.txt file)


In [58]:
# ============================================================
# Load Selected Job Description
# ============================================================

if choice == "1":

    jd_number = int(input("Enter the Job Description number: "))

    selected_jd = jd_files[jd_number - 1]

elif choice == "2":

    custom_path = input("Enter the full path of your Job Description (.txt): ").strip().strip('"')

    selected_jd = Path(custom_path)

    if not selected_jd.exists():
        raise FileNotFoundError(f"File not found: {selected_jd}")

else:
    raise ValueError("Invalid choice.")

print(f"\nSelected Job Description: {selected_jd.name}")


Selected Job Description: jd_devops_cloud_architect.txt


In [59]:
# ============================================================
# Read Job Description
# ============================================================

with open(selected_jd, "r", encoding="utf-8") as file:
    job_description = file.read()

print(job_description)

JOB TITLE: Senior DevOps & Cloud Architect
LOCATION: Austin, TX / Remote
EXPERIENCE REQUIRED: 6+ years

JOB DESCRIPTION:
Join our cloud operations team to build scalable, automated Infrastructure-as-Code. You will manage multi-region Kubernetes clusters on AWS, automate CI/CD pipelines, and monitor telemetry with Prometheus/Grafana.

MUST-HAVE REQUIREMENTS:
- 6+ years of DevOps / Infrastructure experience.
- Mastery of Terraform, Ansible, and Infrastructure as Code (IaC).
- Heavy hands-on experience with Kubernetes (EKS/GKE) and Docker.
- Advanced scripting in Python or Bash.
- Deep AWS expertise (VPC, IAM, EKS, EC2).

NICE-TO-HAVE SKILLS:
- GitOps experience with ArgoCD or Flux.
- Certified AWS Solutions Architect or CKA certification.



# Step 8: Semantic Resume Retrieval

The objective of semantic retrieval is to identify the resume chunks that are most relevant to the given job description.

Unlike traditional keyword-based search, semantic retrieval compares the **meaning** of the job description with the meaning of each resume chunk using vector embeddings.

In this step:

- The job description is converted into an embedding.
- ChromaDB performs similarity search over all indexed resume chunks.
- The Top-K most relevant chunks are retrieved along with their similarity scores.

These retrieved chunks will serve as the input for metadata extraction, hybrid search, and candidate ranking.

In [66]:
# ============================================================
# Semantic Search
# ============================================================

TOP_K = 10

semantic_results = vector_store.similarity_search_with_score(
    query=job_description,
    k=TOP_K
)

print(f"Retrieved {len(semantic_results)} matching resume chunks.")

Retrieved 10 matching resume chunks.


In [67]:
# ============================================================
# Inspect Retrieved Results
# ============================================================

document, score = semantic_results[0]

print("Similarity Score :", score)

print("\nMetadata")

print(document.metadata)

print("\nPreview\n")

print(document.page_content[:800])

Similarity Score : 0.3658221662044525

Metadata
{'page_label': '1', 'producer': 'Microsoft® Word 2019', 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\emily_davis.pdf', 'moddate': '2026-08-08T01:34:27+05:30', 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\emily_davis.pdf', 'total_pages': 1, 'page': 0, 'resume_name': 'emily_davis', 'creationdate': '2026-08-08T01:34:27+05:30', 'author': 'Mohankumar Markuli Chandrayigowda', 'creator': 'Microsoft® Word 2019'}

Preview

EMILY DAVIS 
Email: emily.davis@cloudworks.io | Phone: +1-555-0104 
Location: Austin, TX 
 
PROFESSIONAL SUMMARY 
Senior DevOps & Cloud Architect with 8 years of experience building resilient AWS & 
Kubernetes infrastructure. 
 
TECHNICAL SKILLS 
Terraform, Kubernetes, Docker, AWS, Python, Bash, Ansible, Prometheus, Grafana, Jenkins, 
GitLab CI, ArgoCD 
 
EDUCATION 
Bachelor of Science in Computer Information Systems, UT Austin (2016) 
 
PROFESSIONAL EXPERIENCE 
Senior DevOps Archit

In [68]:
# ============================================================
# Display Top-K Results
# ============================================================

for rank, (document, score) in enumerate(semantic_results, start=1):

    print("=" * 100)

    print(f"Rank              : {rank}")
    print(f"Resume            : {document.metadata['resume_name']}")
    print(f"Page              : {document.metadata['page']}")
    print(f"Similarity Score  : {score:.4f}")

    print("\nPreview\n")

    print(document.page_content[:400])

Rank              : 1
Resume            : emily_davis
Page              : 0
Similarity Score  : 0.3658

Preview

EMILY DAVIS 
Email: emily.davis@cloudworks.io | Phone: +1-555-0104 
Location: Austin, TX 
 
PROFESSIONAL SUMMARY 
Senior DevOps & Cloud Architect with 8 years of experience building resilient AWS & 
Kubernetes infrastructure. 
 
TECHNICAL SKILLS 
Terraform, Kubernetes, Docker, AWS, Python, Bash, Ansible, Prometheus, Grafana, Jenkins, 
GitLab CI, ArgoCD 
 
EDUCATION 
Bachelor of Science in Computer
Rank              : 2
Resume            : marcus_lewis
Page              : 0
Similarity Score  : 0.4939

Preview

MARCUS LEWIS 
Email: marcus.lewis18@example.com | Phone: +1-555-0118 
Location: San Francisco, CA 
 
PROFESSIONAL SUMMARY 
Experienced DevOps Engineer with 10 years of hands-on industry experience building 
scalable systems and data products. 
 
TECHNICAL SKILLS 
Docker, Kubernetes, Terraform, AWS, CI/CD, Python, Bash, Prometheus, ArgoCD, REST APIs, 
Unit Testing, Linux

In [71]:
retriever.invoke(job_description)

[Document(id='696351ac-8735-4423-9f1a-9a1d90cb237a', metadata={'producer': 'Microsoft® Word 2019', 'creationdate': '2026-08-08T01:34:27+05:30', 'page_label': '1', 'page': 0, 'author': 'Mohankumar Markuli Chandrayigowda', 'resume_name': 'emily_davis', 'creator': 'Microsoft® Word 2019', 'total_pages': 1, 'source': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\emily_davis.pdf', 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\emily_davis.pdf', 'moddate': '2026-08-08T01:34:27+05:30'}, page_content='EMILY DAVIS \nEmail: emily.davis@cloudworks.io | Phone: +1-555-0104 \nLocation: Austin, TX \n \nPROFESSIONAL SUMMARY \nSenior DevOps & Cloud Architect with 8 years of experience building resilient AWS & \nKubernetes infrastructure. \n \nTECHNICAL SKILLS \nTerraform, Kubernetes, Docker, AWS, Python, Bash, Ansible, Prometheus, Grafana, Jenkins, \nGitLab CI, ArgoCD \n \nEDUCATION \nBachelor of Science in Computer Information Systems, UT Austin (2016) \n \nPROFESSIONA

# Extract Candidate Metadata

To improve candidate ranking and filtering, we extract structured information from the retrieved resume chunks.

The extracted metadata includes:

- Candidate Name
- Technical Skills
- Years of Experience
- Highest Education

This metadata will be used in later stages for:

- Hybrid Search
- Must-have Skill Filtering
- Candidate Ranking
- Match Scoring
- JSON Output

Instead of relying on regular expressions, we use the Large Language Model to extract structured information in JSON format, making the solution more robust across different resume formats.

In [76]:
from langchain_core.prompts import ChatPromptTemplate

metadata_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert resume parser.

Extract:
- candidate_name
- skills
- experience_years
- education

Return ONLY valid JSON.
"""
    ),
    (
        "human",
        "{resume}"
    )
])

In [77]:
import json

candidate_metadata = []

for document, score in semantic_results:

    try:

        prompt = metadata_prompt.format_messages(
            resume=document.page_content
        )

        response = llm.invoke(prompt)

        metadata = json.loads(response.content)

        metadata["resume_name"] = document.metadata.get("resume_name")
        metadata["resume_path"] = document.metadata.get("resume_path")
        metadata["semantic_score"] = score

        candidate_metadata.append(metadata)

    except Exception as e:

        print(f"Error processing {document.metadata.get('resume_name')}: {e}")

In [78]:
response_text = response.content.strip()

if response_text.startswith("```"):
    response_text = response_text.replace("```json", "").replace("```", "").strip()

metadata = json.loads(response_text)

# Hybrid Search

Semantic search is excellent at understanding context, but it may not always prioritize exact technical requirements such as programming languages, frameworks, certifications, or tools.

To improve retrieval quality, we combine:

1. **Semantic Search** – Finds resumes with similar meaning using vector embeddings.
2. **Keyword Search (BM25)** – Rewards resumes containing important terms from the job description.

This hybrid approach produces more accurate candidate rankings by leveraging the strengths of both retrieval techniques.

In [79]:
# ============================================================
# Build BM25 Index
# ============================================================

corpus = [doc.page_content for doc in chunked_documents]

tokenized_corpus = [doc.lower().split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

print("✅ BM25 index created.")

✅ BM25 index created.


In [80]:
# ============================================================
# BM25 Search
# ============================================================

query_tokens = job_description.lower().split()

bm25_scores = bm25.get_scores(query_tokens)

print(f"Calculated BM25 scores for {len(bm25_scores)} chunks.")

Calculated BM25 scores for 47 chunks.


In [81]:
# ============================================================
# Top BM25 Results
# ============================================================

import numpy as np

top_indices = np.argsort(bm25_scores)[::-1][:10]

bm25_results = []

for idx in top_indices:
    bm25_results.append(
        (
            chunked_documents[idx],
            bm25_scores[idx]
        )
    )

print(f"Retrieved {len(bm25_results)} BM25 results.")

Retrieved 10 BM25 results.


In [82]:
for rank, (doc, score) in enumerate(bm25_results, start=1):

    print("=" * 80)
    print(f"Rank        : {rank}")
    print(f"Resume      : {doc.metadata['resume_name']}")
    print(f"BM25 Score  : {score:.2f}")

    print("\nPreview:\n")
    print(doc.page_content[:400])

Rank        : 1
Resume      : emily_davis
BM25 Score  : 87.63

Preview:

EMILY DAVIS 
Email: emily.davis@cloudworks.io | Phone: +1-555-0104 
Location: Austin, TX 
 
PROFESSIONAL SUMMARY 
Senior DevOps & Cloud Architect with 8 years of experience building resilient AWS & 
Kubernetes infrastructure. 
 
TECHNICAL SKILLS 
Terraform, Kubernetes, Docker, AWS, Python, Bash, Ansible, Prometheus, Grafana, Jenkins, 
GitLab CI, ArgoCD 
 
EDUCATION 
Bachelor of Science in Computer
Rank        : 2
Resume      : sophia_taylor
BM25 Score  : 33.44

Preview:

SOPHIA TAYLOR 
Email: sophia.taylor@pmhub.com | Phone: +1-555-0106 
Location: New York, NY 
 
PROFESSIONAL SUMMARY 
Principal Technical Product Manager with 7 years leading AI product lifecycle from inception 
to GTM. 
 
TECHNICAL SKILLS 
Product Strategy, Agile/Scrum, A/B Testing, SQL, User Analytics, Jira, Figma, Roadmap 
Planning, LLM Evaluation, Go-To-Market 
 
EDUCATION 
MBA, Columbia Busines
Rank        : 3
Resume      : marcus_lewis
BM25 Sc

# Aggregate Resume Chunks by Candidate

Both semantic search and BM25 search return **individual resume chunks**.

However, recruiters evaluate **candidates**, not isolated sections of their resumes.

Therefore, we aggregate all retrieved chunks belonging to the same candidate.

For each candidate, we retain:

- Candidate Name
- Resume Path
- Semantic Similarity Score
- BM25 Keyword Score
- Relevant Resume Excerpts

This aggregated representation forms the basis for the final candidate ranking.

In [83]:
from collections import defaultdict

candidate_results = defaultdict(
    lambda: {
        "resume_name": "",
        "resume_path": "",
        "semantic_score": 0,
        "bm25_score": 0,
        "chunks": []
    }
)

for document, score in semantic_results:

    resume_name = document.metadata["resume_name"]

    candidate_results[resume_name]["resume_name"] = resume_name

    candidate_results[resume_name]["resume_path"] = document.metadata["resume_path"]

    # Keep the highest semantic similarity score
    candidate_results[resume_name]["semantic_score"] = max(
        candidate_results[resume_name]["semantic_score"],
        score
    )

    candidate_results[resume_name]["chunks"].append(
        document.page_content
    )

print(f"Candidates Retrieved : {len(candidate_results)}")

Candidates Retrieved : 10


In [84]:
for document, score in bm25_results:

    resume_name = document.metadata["resume_name"]

    if resume_name in candidate_results:

        candidate_results[resume_name]["bm25_score"] = max(
            candidate_results[resume_name]["bm25_score"],
            score
        )

In [85]:
first_candidate = next(iter(candidate_results.values()))

print(first_candidate["resume_name"])

print(first_candidate["semantic_score"])

print(first_candidate["bm25_score"])

print("\nNumber of Relevant Chunks")

print(len(first_candidate["chunks"]))

emily_davis
0.3658221662044525
87.62617055350826

Number of Relevant Chunks
1


# Candidate Ranking and Match Scoring

The retrieved candidates are ranked using a weighted scoring mechanism that combines multiple signals.

Since semantic similarity and BM25 scores are measured on different scales, both scores are first normalized to a common range before calculating the final match score.

### Scoring Components

- **Semantic Similarity (60%)**
  Measures how closely the resume content matches the overall meaning of the job description.

- **Keyword Matching (40%)**
  Rewards resumes containing important technologies, programming languages, frameworks, and domain-specific keywords.

The final score is scaled to a **0–100** range for easier interpretation.

In [86]:
import pandas as pd

ranking_df = pd.DataFrame(candidate_results.values())

ranking_df.head()

,resume_name,resume_path,semantic_score,bm25_score,chunks
0,emily_davis,d:\GitHub\RAG-Based-Profile-matching\data\resu...,0.365822,87.626171,[EMILY DAVIS \nEmail: emily.davis@cloudworks.i...
1,marcus_lewis,d:\GitHub\RAG-Based-Profile-matching\data\resu...,0.493865,31.574019,[MARCUS LEWIS \nEmail: marcus.lewis18@example....
2,kevin_allen,d:\GitHub\RAG-Based-Profile-matching\data\resu...,0.708492,0.000000,[KEVIN ALLEN \nEmail: kevin.allen27@example.co...
3,justin_lee,d:\GitHub\RAG-Based-Profile-matching\data\resu...,0.740842,0.000000,[JUSTIN LEE \nEmail: justin.lee26@example.com ...
4,alex_green,d:\GitHub\RAG-Based-Profile-matching\data\resu...,0.758024,20.370474,[ALEX GREEN \nEmail: alex.green14@example.com ...


In [87]:
# Normalize Semantic Score
semantic_min = ranking_df["semantic_score"].min()
semantic_max = ranking_df["semantic_score"].max()

ranking_df["semantic_norm"] = (
    (ranking_df["semantic_score"] - semantic_min)
    /
    (semantic_max - semantic_min + 1e-9)
)

# Normalize BM25 Score
bm25_min = ranking_df["bm25_score"].min()
bm25_max = ranking_df["bm25_score"].max()

ranking_df["bm25_norm"] = (
    (ranking_df["bm25_score"] - bm25_min)
    /
    (bm25_max - bm25_min + 1e-9)
)

In [89]:
# Compute Final Match Score
ranking_df["match_score"] = (
    0.60 * ranking_df["semantic_norm"]
    +
    0.40 * ranking_df["bm25_norm"]
) * 100

ranking_df["match_score"] = ranking_df["match_score"].round(2)

In [90]:
# Sort Candidates
ranking_df = ranking_df.sort_values(
    by="match_score",
    ascending=False
).reset_index(drop=True)

ranking_df[
    [
        "resume_name",
        "semantic_score",
        "bm25_score",
        "match_score"
    ]
]

,resume_name,semantic_score,bm25_score,match_score
0,daniel_clark,0.792291,21.119768,68.95
1,alex_green,0.758024,20.370474,63.84
2,ethan_young,0.797292,0.000000,60.00
3,elena_lewis,0.785591,0.000000,58.37
4,christopher_hall,0.773874,0.000000,56.74
5,laura_hill,0.768858,0.000000,56.05
6,justin_lee,0.740842,0.000000,52.15
7,kevin_allen,0.708492,0.000000,47.65
8,emily_davis,0.365822,87.626171,40.00
9,marcus_lewis,0.493865,31.574019,32.22


In [91]:
ranking_df["semantic_score"] = 1 / (1 + ranking_df["semantic_score"])

In [92]:
semantic_results[0][1]

0.3658221662044525

# Generate Match Reasoning

After ranking the candidates, the system generates a human-readable explanation describing why each candidate matches the job description.

The Large Language Model analyzes:

- The job description
- Candidate metadata
- Relevant resume excerpts
- Match score

It then summarizes the strengths of each candidate, highlighting matching skills, experience, and qualifications.

These explanations improve transparency and help recruiters understand the ranking results.

In [93]:
from langchain_core.prompts import ChatPromptTemplate

reasoning_prompt = ChatPromptTemplate.from_template("""
You are an expert technical recruiter.

Job Description:
{job_description}

Candidate Name:
{candidate_name}

Match Score:
{match_score}

Resume Excerpts:
{resume_excerpt}

Provide:

1. Matching Skills
2. Matching Experience
3. Strengths
4. Missing Skills (if any)
5. A concise reasoning in 3-4 sentences.

Return ONLY valid JSON.

{{
    "matched_skills": [],
    "missing_skills": [],
    "reasoning": ""
}}
""")

In [94]:
import json

candidate_reasoning = []

TOP_RESULTS = 10

for _, candidate in ranking_df.head(TOP_RESULTS).iterrows():

    excerpt = "\n\n".join(candidate["chunks"][:3])

    prompt = reasoning_prompt.format_messages(
        job_description=job_description,
        candidate_name=candidate["resume_name"],
        match_score=candidate["match_score"],
        resume_excerpt=excerpt
    )

    response = llm.invoke(prompt)

    try:
        result = json.loads(
            response.content.replace("```json", "").replace("```", "").strip()
        )

    except Exception:

        result = {
            "matched_skills": [],
            "missing_skills": [],
            "reasoning": response.content
        }

    result["resume_name"] = candidate["resume_name"]
    result["resume_path"] = candidate["resume_path"]
    result["match_score"] = candidate["match_score"]

    candidate_reasoning.append(result)

print("✅ Match reasoning generated.")

✅ Match reasoning generated.


In [95]:
candidate_reasoning[0]

{'matched_skills': [],
 'missing_skills': [],
 'reasoning': '```json\n{\n    "matched_skills": [\n        "Python",\n        "Docker",\n        "CI/CD",\n        "AWS"\n    ],\n    "missing_skills": [\n        "Terraform",\n        "Ansible",\n        "Kubernetes",\n        "Infrastructure as Code (IaC)",\n        "Advanced scripting in Bash",\n        "GitOps experience",\n        "AWS Solutions Architect certification",\n        "CKA certification"\n    ],\n    "reasoning": "Daniel Clark has a solid foundation in Python, Docker, and CI/CD, which are relevant to the DevOps role. However, he lacks critical skills such as Terraform, Ansible, and Kubernetes experience, which are essential for the position. Additionally, he does not have the required',
 'resume_name': 'daniel_clark',
 'resume_path': 'd:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\daniel_clark.pdf',
 'match_score': 68.95}

#  Generate the Final Job Matching Results

The final stage of the Resume RAG pipeline is to generate a structured JSON response containing the ranked candidates.

The output follows the assignment specification and includes:

- Job Description
- Candidate Name
- Resume Path
- Match Score (0–100)
- Matched Skills
- Relevant Resume Excerpts
- Match Reasoning

This JSON can be easily consumed by downstream applications such as Applicant Tracking Systems (ATS), HR dashboards, or recruitment portals.

In [96]:
# ============================================================
# Generate Final Output
# ============================================================

final_output = {
    "job_description": job_description,
    "top_matches": []
}

for candidate in candidate_reasoning:

    # Find candidate details
    row = ranking_df[
        ranking_df["resume_name"] == candidate["resume_name"]
    ].iloc[0]

    final_output["top_matches"].append({

        "candidate_name": candidate["resume_name"],

        "resume_path": candidate["resume_path"],

        "match_score": round(candidate["match_score"], 2),

        "matched_skills": candidate["matched_skills"],

        "relevant_excerpts": row["chunks"][:2],

        "reasoning": candidate["reasoning"]

    })

In [97]:
import json

print(
    json.dumps(
        final_output,
        indent=4
    )
)

{
    "job_description": "JOB TITLE: Senior DevOps & Cloud Architect\nLOCATION: Austin, TX / Remote\nEXPERIENCE REQUIRED: 6+ years\n\nJOB DESCRIPTION:\nJoin our cloud operations team to build scalable, automated Infrastructure-as-Code. You will manage multi-region Kubernetes clusters on AWS, automate CI/CD pipelines, and monitor telemetry with Prometheus/Grafana.\n\nMUST-HAVE REQUIREMENTS:\n- 6+ years of DevOps / Infrastructure experience.\n- Mastery of Terraform, Ansible, and Infrastructure as Code (IaC).\n- Heavy hands-on experience with Kubernetes (EKS/GKE) and Docker.\n- Advanced scripting in Python or Bash.\n- Deep AWS expertise (VPC, IAM, EKS, EC2).\n\nNICE-TO-HAVE SKILLS:\n- GitOps experience with ArgoCD or Flux.\n- Certified AWS Solutions Architect or CKA certification.\n",
    "top_matches": [
        {
            "candidate_name": "daniel_clark",
            "resume_path": "d:\\GitHub\\RAG-Based-Profile-matching\\data\\resumes\\daniel_clark.pdf",
            "match_score": 6

In [98]:
# ============================================================
# Save Results
# ============================================================

output_file = OUTPUT_DIR / "job_matching_results.json"

with open(output_file, "w", encoding="utf-8") as file:

    json.dump(
        final_output,
        file,
        indent=4
    )

print(f"Results saved to:\n{output_file}")

Results saved to:
d:\GitHub\RAG-Based-Profile-matching\outputs\job_matching_results.json


# Performance Evaluation

To evaluate the effectiveness of the Resume RAG System, we measure several performance metrics.

### Metrics

- Number of Resume Documents
- Number of Resume Chunks
- Number of Indexed Embeddings
- Number of Retrieved Candidates
- Retrieval Latency
- Average Match Score

These metrics provide insight into the scalability and efficiency of the retrieval pipeline.

In [99]:
import time

start = time.perf_counter()

_ = vector_store.similarity_search(
    job_description,
    k=10
)

end = time.perf_counter()

latency = end - start

print("=" * 60)

print(f"Total Resumes        : {len(resume_files)}")

print(f"Total Chunks         : {len(chunked_documents)}")

print(f"Retrieved Candidates : {len(candidate_reasoning)}")

print(f"Retrieval Time       : {latency:.4f} seconds")

print(f"Average Match Score  : {ranking_df['match_score'].mean():.2f}")

print("=" * 60)

Total Resumes        : 34
Total Chunks         : 47
Retrieved Candidates : 10
Retrieval Time       : 0.3671 seconds
Average Match Score  : 53.60


# Conclusion

In this project, we implemented a complete Retrieval-Augmented Generation (RAG) system for intelligent resume screening and job matching.

## Key Features

- Loaded and processed resume PDF documents.
- Applied intelligent document chunking.
- Generated semantic embeddings.
- Indexed resume chunks using ChromaDB.
- Retrieved relevant resumes using semantic search.
- Enhanced retrieval using Hybrid Search (Semantic + BM25).
- Ranked candidates using a weighted scoring mechanism.
- Generated explainable match reasoning using a Large Language Model.
- Produced structured JSON output compatible with recruitment systems.

This implementation demonstrates how modern RAG architectures can improve recruitment workflows by combining semantic retrieval, keyword matching, and LLM-powered reasoning.